In [10]:
import numpy as np
import pandas as pd

# Load dataset
df = pd.read_csv("../data/raw/Medical_Device_Failure_dataset.csv")

print("Shape:", df.shape)
df.head()

Shape: (4149, 13)


,Device_ID,Device_Type,Purchase_Date,Age,Manufacturer,Model,Country,Maintenance_Cost,Downtime,Maintenance_Frequency,Failure_Event_Count,Maintenance_Class,Maintenance_Report
0,MD03449,Defibrillator,2018-04-23,7,CardioSync,Model-100,France,7115.349585,7.933824,3,0,1,Component component upgrade after detecting ov...
1,MD02024,Infusion Pump,2020-12-10,5,MedEquip,Model-650,Italy,7290.780658,7.838711,3,4,2,battery wear caused operational delay; replace...
2,MD04239,MRI Scanner,2023-11-22,2,ImagingTech,Model-650,France,5635.521788,13.911045,1,2,3,data lag caused operational delay; component u...
3,MD00153,Defibrillator,2021-03-03,4,RescueTech,Model-450,UK,5001.360188,29.059510,3,1,3,Routine check completed; battery wear observed...
4,MD03743,Defibrillator,2019-05-16,6,RescueTech,Model-450,Canada,7555.132928,13.942355,4,4,2,Component inspection after detecting voltage s...


In [11]:
# -----------------------------
# Phase 1.1: Missing Value Handling
# -----------------------------

clean_df = df.copy()

num_cols = clean_df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = clean_df.select_dtypes(exclude=[np.number]).columns.tolist()

# Median imputation for numerical columns
for col in num_cols:
    clean_df[col] = clean_df[col].fillna(clean_df[col].median())

# Mode imputation for categorical columns
for col in cat_cols:
    if clean_df[col].isna().any():
        mode_val = clean_df[col].mode(dropna=True)
        fill_val = mode_val.iloc[0] if not mode_val.empty else "Unknown"
        clean_df[col] = clean_df[col].fillna(fill_val)

print("Missing values after imputation:")
print(clean_df.isna().sum().sort_values(ascending=False).head(10))

Missing values after imputation:
Device_ID                0
Device_Type              0
Purchase_Date            0
Age                      0
Manufacturer             0
Model                    0
Country                  0
Maintenance_Cost         0
Downtime                 0
Maintenance_Frequency    0
dtype: int64


In [12]:
# -----------------------------
# Phase 1.2: Outlier Capping (IQR Method)
# -----------------------------
# We cap values to IQR bounds (winsorization) to avoid dropping too much data.

def iqr_cap(series, k=1.5):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - k * iqr
    upper = q3 + k * iqr
    return series.clip(lower, upper)

pre_outlier_shape = clean_df.shape

for col in num_cols:
    clean_df[col] = iqr_cap(clean_df[col])

print("Shape before outlier handling:", pre_outlier_shape)
print("Shape after outlier handling:", clean_df.shape)

# Optional rule-based impossible stat check example (if columns exist)
if {"Operational_Hours", "Device_Age_Years"}.issubset(clean_df.columns):
    impossible_mask = (
        (clean_df["Device_Age_Years"] > 0)
        & (clean_df["Operational_Hours"] > clean_df["Device_Age_Years"] * 24 * 365)
    )
    print("Potentially impossible records found:", int(impossible_mask.sum()))

Shape before outlier handling: (4149, 13)
Shape after outlier handling: (4149, 13)


In [13]:
# -----------------------------
# Phase 1.3: Categorical Encoding
# -----------------------------
# One-Hot encode categorical fields for model-ready numeric input.

encoded_df = pd.get_dummies(clean_df, columns=cat_cols, drop_first=True)

print("Original shape:", clean_df.shape)
print("Encoded shape:", encoded_df.shape)
encoded_df.head()

Original shape: (4149, 13)
Encoded shape: (4149, 10097)


,Age,Maintenance_Cost,Downtime,Maintenance_Frequency,Failure_Event_Count,Maintenance_Class,Device_ID_MD00003,Device_ID_MD00006,Device_ID_MD00007,Device_ID_MD00008,...,Maintenance_Report_voltage spike caused operational delay; system reset initiated. data loss hardware fault,Maintenance_Report_voltage spike caused operational delay; system reset initiated. data loss high priority failure,Maintenance_Report_voltage spike caused operational delay; system reset initiated. data loss system crash,Maintenance_Report_voltage spike caused operational delay; system reset initiated. log cleaned low priority fault,Maintenance_Report_voltage spike caused operational delay; system reset initiated. log cleaned routine reset,Maintenance_Report_voltage spike caused operational delay; system reset initiated. log cleaned temporary glitch,Maintenance_Report_voltage spike caused operational delay; system reset initiated. moderate delay average severity report,Maintenance_Report_voltage spike caused operational delay; system reset initiated. reviewed logs driver update applied,Maintenance_Report_voltage spike caused operational delay; system reset initiated. reviewed logs non-critical warning,Maintenance_Report_voltage spike caused operational delay; system reset initiated. sensor replaced urgent inspection needed
0,7,7115.349585,7.933824,3,0,1,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,5,7290.780658,7.838711,3,4,2,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,2,5635.521788,13.911045,1,2,3,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,4,5001.360188,28.996465,3,1,3,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,6,7555.132928,13.942355,4,4,2,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [14]:
# -----------------------------
# Phase 2: Feature Engineering (Derived Columns)
# -----------------------------

fe_df = clean_df.copy()

# Helper to pick first available column name from common variants
def pick_col(df_, candidates):
    for c in candidates:
        if c in df_.columns:
            return c
    return None

# Resolve source columns from your dataset
failure_count_col = pick_col(fe_df, ["Failure_Event_Count", "FailureCount", "Failure_Events"])
maintenance_cost_col = pick_col(fe_df, ["Maintenance_Cost", "MaintenanceCost", "Repair_Cost"])
age_col = pick_col(fe_df, ["Current_Age", "Device_Age_Years", "Age"])
operational_hours_col = pick_col(fe_df, ["Operational_Hours", "OperationalHours", "Hours_Used"])
expected_life_col = pick_col(fe_df, ["Expected_Lifespan", "Expected_Life_Years", "Lifespan"])

# Safe divide helper: avoids inf values and divide-by-zero errors
def safe_divide(numerator, denominator):
    result = numerator / denominator.replace(0, np.nan)
    return result.replace([np.inf, -np.inf], np.nan)

# If Operational_Hours is missing, estimate from Age (years -> hours)
if operational_hours_col is None and age_col is not None:
    fe_df["Operational_Hours_Est"] = fe_df[age_col] * 365 * 24
    operational_hours_col = "Operational_Hours_Est"
    print("Using estimated operational hours from Age.")

# If Expected_Lifespan is missing, estimate by device type max age (fallback: global max age)
if expected_life_col is None and age_col is not None:
    if "Device_Type" in fe_df.columns:
        life_by_type = fe_df.groupby("Device_Type")[age_col].transform("max").replace(0, np.nan)
        fe_df["Expected_Lifespan_Est"] = life_by_type.fillna(fe_df[age_col].max())
    else:
        fe_df["Expected_Lifespan_Est"] = fe_df[age_col].max()
    expected_life_col = "Expected_Lifespan_Est"
    print("Using estimated expected lifespan.")

# MTBF = Operational_Hours / Failure_Event_Count
if operational_hours_col and failure_count_col:
    fe_df["MTBF"] = safe_divide(fe_df[operational_hours_col], fe_df[failure_count_col])
else:
    print("Skipping MTBF: required columns not found.")

# Cost_Per_Hour = Maintenance_Cost / Operational_Hours
if maintenance_cost_col and operational_hours_col:
    fe_df["Cost_Per_Hour"] = safe_divide(fe_df[maintenance_cost_col], fe_df[operational_hours_col])
else:
    print("Skipping Cost_Per_Hour: required columns not found.")

# Lifespan_Usage_Ratio = Current_Age / Expected_Lifespan
if age_col and expected_life_col:
    fe_df["Lifespan_Usage_Ratio"] = safe_divide(fe_df[age_col], fe_df[expected_life_col])
else:
    print("Skipping Lifespan_Usage_Ratio: required columns not found.")

# Fill NaNs created by safe division with median for stability
new_feature_cols = [c for c in ["MTBF", "Cost_Per_Hour", "Lifespan_Usage_Ratio"] if c in fe_df.columns]
for col in new_feature_cols:
    fe_df[col] = fe_df[col].fillna(fe_df[col].median())

print("Added feature columns:", new_feature_cols)
fe_df[new_feature_cols].describe().T if new_feature_cols else fe_df.head()

Using estimated operational hours from Age.
Using estimated expected lifespan.
Added feature columns: ['MTBF', 'Cost_Per_Hour', 'Lifespan_Usage_Ratio']


,count,mean,std,min,25%,50%,75%,max
MTBF,4149.0,32934.278139,23268.668748,1460.000000,17520.000000,26280.000000,43800.00000,105120.000000
Cost_Per_Hour,4149.0,0.209101,0.244445,-0.283584,0.069475,0.136222,0.25463,2.669192
Lifespan_Usage_Ratio,4149.0,0.559814,0.264634,0.083333,0.333333,0.583333,0.75000,1.000000


In [15]:
# -----------------------------
# Phase 3: Synthetic Time-Series Generation (IoT Simulator)
# -----------------------------
# Generates 365 daily logs per device, anchored to each device's failure count.

if "fe_df" not in globals():
    raise ValueError("`fe_df` not found. Run Phase 1 and Phase 2 first.")

rng = np.random.default_rng(42)
base_df = fe_df.copy()

required_cols = ["Device_ID", "Failure_Event_Count"]
for c in required_cols:
    if c not in base_df.columns:
        raise ValueError(f"Missing required column: {c}")

age_col = "Age" if "Age" in base_df.columns else None
report_col = "Maintenance_Report" if "Maintenance_Report" in base_df.columns else None
device_type_col = "Device_Type" if "Device_Type" in base_df.columns else None

sim_end = pd.Timestamp.today().normalize()
sim_start = sim_end - pd.Timedelta(days=364)
all_days = pd.date_range(sim_start, sim_end, freq="D")

ELECTRICAL_KWS = ["voltage", "circuit", "power", "electrical"]
DEVICE_TYPE_BASELINES = {
    # temp_var, vibration_hz, voltage_drop
    "MRI Scanner": (0.85, 17.0, 0.70),
    "CT Scanner": (0.80, 20.0, 0.65),
    "X-Ray Machine": (0.75, 23.0, 0.60),
    "PET Scanner": (0.78, 21.0, 0.62),
    "Defibrillator": (0.68, 26.0, 0.50),
    "Infusion Pump": (0.62, 27.5, 0.45),
}

def has_electrical_pattern(text):
    if not isinstance(text, str):
        return False
    t = text.lower()
    return any(k in t for k in ELECTRICAL_KWS)

def schedule_failure_days(n_failures, n_days=365):
    n = int(max(0, n_failures))
    if n == 0:
        return []
    n = min(n, n_days)
    anchors = np.linspace(30, n_days - 1, n).astype(int)
    jitter = rng.integers(-4, 5, size=n)
    days = np.clip(anchors + jitter, 0, n_days - 1)
    return sorted(np.unique(days).tolist())

def next_failure_in_hours(day_idx, failure_idxs):
    future = [f for f in failure_idxs if f >= day_idx]
    if not future:
        return np.nan
    return float((future[0] - day_idx) * 24)

def is_near_failure(days_to_next, near_window=10):
    return pd.notna(days_to_next) and days_to_next <= near_window

rows = []

for _, dev in base_df.iterrows():
    device_id = dev["Device_ID"]
    device_type = dev[device_type_col] if device_type_col else "Unknown"
    failure_count = dev["Failure_Event_Count"]
    is_electrical = has_electrical_pattern(dev[report_col]) if report_col else False

    age_factor = float(dev[age_col]) if age_col else 5.0
    wear = 1.0 + (age_factor / 20.0)

    d_temp, d_vib, d_volt = DEVICE_TYPE_BASELINES.get(device_type, (0.70, 25.0, 0.50))
    temp_base = rng.normal(d_temp, 0.10) * wear
    vib_base = rng.normal(d_vib, 2.0) * wear
    volt_base = rng.normal(d_volt, 0.08) * wear

    failure_day_idxs = schedule_failure_days(failure_count, n_days=len(all_days))

    for i, ts in enumerate(all_days):
        future = [f for f in failure_day_idxs if f >= i]
        days_to_next = (future[0] - i) if future else np.nan

        temp = temp_base + rng.normal(0, 0.08)
        vib = vib_base + rng.normal(0, 0.9)
        vdrop = max(0.0, volt_base + rng.normal(0, 0.05))

        if pd.notna(days_to_next):
            if days_to_next <= 10:
                temp += (10 - days_to_next) * rng.uniform(0.08, 0.18)
            if days_to_next <= 7:
                vib += (7 - days_to_next + 1) * rng.uniform(0.7, 1.4)
                prob_spike = 0.45 if is_electrical else 0.20
                if rng.random() < prob_spike:
                    vdrop += rng.uniform(0.35, 1.1 if is_electrical else 0.7)

        # False flags: one-day anomalies away from failure windows
        if not is_near_failure(days_to_next, near_window=10):
            if rng.random() < 0.015:
                temp += rng.uniform(0.4, 1.2)
            if rng.random() < 0.020:
                vib += rng.uniform(4.0, 11.0)

        temp = max(0.01, temp)
        vib = max(0.01, vib)

        rul_hours = next_failure_in_hours(i, failure_day_idxs)
        rul_days = (rul_hours / 24.0) if pd.notna(rul_hours) else np.nan

        rows.append(
            {
                "Device_ID": device_id,
                "Device_Type": device_type,
                "Timestamp": ts,
                "Simulated_Temperature_Variance": float(temp),
                "Simulated_Motor_Vibration_Hz": float(vib),
                "Simulated_Voltage_Drop": float(vdrop),
                "RUL_Hours": rul_hours,
                "RUL_Days": rul_days,
                "RUL": rul_days,
            }
        )

timeseries_df = pd.DataFrame(rows)
timeseries_df["RUL_Hours"] = timeseries_df["RUL_Hours"].fillna(9999.0)
timeseries_df["RUL_Days"] = timeseries_df["RUL_Days"].fillna(9999.0 / 24.0)
timeseries_df["RUL"] = timeseries_df["RUL"].fillna(9999.0 / 24.0)

print("Synthetic time-series shape:", timeseries_df.shape)
print("Columns:", timeseries_df.columns.tolist())
timeseries_df.head()

Synthetic time-series shape: (1514385, 9)
Columns: ['Device_ID', 'Device_Type', 'Timestamp', 'Simulated_Temperature_Variance', 'Simulated_Motor_Vibration_Hz', 'Simulated_Voltage_Drop', 'RUL_Hours', 'RUL_Days', 'RUL']


,Device_ID,Device_Type,Timestamp,Simulated_Temperature_Variance,Simulated_Motor_Vibration_Hz,Simulated_Voltage_Drop,RUL_Hours,RUL_Days,RUL
0,MD03449,Defibrillator,2025-04-11,1.034382,30.536111,0.690940,9999.0,416.625,416.625
1,MD03449,Defibrillator,2025-04-12,0.957793,31.524303,0.800019,9999.0,416.625,416.625
2,MD03449,Defibrillator,2025-04-13,1.049316,32.712801,0.713084,9999.0,416.625,416.625
3,MD03449,Defibrillator,2025-04-14,1.029413,32.247110,0.746806,9999.0,416.625,416.625
4,MD03449,Defibrillator,2025-04-15,0.946774,31.906548,0.738442,9999.0,416.625,416.625


In [16]:
# -----------------------------
# Phase 4: Target Variable Definition (Labels)
# -----------------------------

if "fe_df" not in globals():
    raise ValueError("`fe_df` not found. Run Phase 1 and Phase 2 first.")
if "timeseries_df" not in globals():
    raise ValueError("`timeseries_df` not found. Run Phase 3 first.")

labeled_device_df = fe_df.copy()

# ---- Target 1: Risk_Class (for Logistic Regression) ----
# Rules:
# - zero failures -> Low Risk
# - low MTBF & high Maintenance_Cost -> High Risk
# - otherwise -> Medium Risk

if "MTBF" not in labeled_device_df.columns:
    # fallback for robustness
    if {"Failure_Event_Count", "Age"}.issubset(labeled_device_df.columns):
        est_hours = labeled_device_df["Age"] * 365 * 24
        labeled_device_df["MTBF"] = est_hours / labeled_device_df["Failure_Event_Count"].replace(0, np.nan)
        labeled_device_df["MTBF"] = labeled_device_df["MTBF"].replace([np.inf, -np.inf], np.nan).fillna(est_hours.median())
    else:
        raise ValueError("Cannot compute MTBF: required columns missing.")

if "Maintenance_Cost" not in labeled_device_df.columns:
    raise ValueError("`Maintenance_Cost` is required for Risk_Class logic.")

mtbf_low_thr = labeled_device_df["MTBF"].quantile(0.33)
cost_high_thr = labeled_device_df["Maintenance_Cost"].quantile(0.67)

failure_col = "Failure_Event_Count"
if failure_col not in labeled_device_df.columns:
    raise ValueError("`Failure_Event_Count` is required for Risk_Class logic.")

def assign_risk(row):
    if row[failure_col] == 0:
        return "Low Risk"
    if (row["MTBF"] <= mtbf_low_thr) and (row["Maintenance_Cost"] >= cost_high_thr):
        return "High Risk"
    return "Medium Risk"

labeled_device_df["Risk_Class"] = labeled_device_df.apply(assign_risk, axis=1)

# Optional numeric version for ML pipelines that require numeric labels
risk_map = {"Low Risk": 0, "Medium Risk": 1, "High Risk": 2}
labeled_device_df["Risk_Class_Label"] = labeled_device_df["Risk_Class"].map(risk_map)

# ---- Target 2: Will_Fail_In_72_Hours (for RF/LSTM) ----
labeled_ts_df = timeseries_df.copy()
labeled_ts_df["Will_Fail_In_72_Hours"] = (labeled_ts_df["RUL_Hours"] <= 72).astype(int)

print("Risk_Class distribution:")
print(labeled_device_df["Risk_Class"].value_counts())

print("\nWill_Fail_In_72_Hours distribution:")
print(labeled_ts_df["Will_Fail_In_72_Hours"].value_counts())

print("\nDevice-level labeled preview:")
display(labeled_device_df[["Device_ID", "Failure_Event_Count", "MTBF", "Maintenance_Cost", "Risk_Class", "Risk_Class_Label"]].head())

print("\nTime-series labeled preview:")
display(labeled_ts_df[["Device_ID", "Timestamp", "RUL_Hours", "RUL", "Will_Fail_In_72_Hours"]].head())

Risk_Class distribution:
Risk_Class
Medium Risk    3082
Low Risk        588
High Risk       479
Name: count, dtype: int64

Will_Fail_In_72_Hours distribution:
Will_Fail_In_72_Hours
0    1481997
1      32388
Name: count, dtype: int64

Device-level labeled preview:


,Device_ID,Failure_Event_Count,MTBF,Maintenance_Cost,Risk_Class,Risk_Class_Label
0,MD03449,0,26280.0,7115.349585,Low Risk,0
1,MD02024,4,10950.0,7290.780658,Medium Risk,1
2,MD04239,2,8760.0,5635.521788,Medium Risk,1
3,MD00153,1,35040.0,5001.360188,Medium Risk,1
4,MD03743,4,13140.0,7555.132928,Medium Risk,1



Time-series labeled preview:


,Device_ID,Timestamp,RUL_Hours,RUL,Will_Fail_In_72_Hours
0,MD03449,2025-04-11,9999.0,416.625,0
1,MD03449,2025-04-12,9999.0,416.625,0
2,MD03449,2025-04-13,9999.0,416.625,0
3,MD03449,2025-04-14,9999.0,416.625,0
4,MD03449,2025-04-15,9999.0,416.625,0


In [18]:
# -----------------------------
# Export processed datasets to CSV
# -----------------------------

if "labeled_device_df" not in globals():
    raise ValueError("`labeled_device_df` not found. Run Phase 4 first.")
if "labeled_ts_df" not in globals():
    raise ValueError("`labeled_ts_df` not found. Run Phase 4 first.")

device_out = "../data/processed/processed_device_level.csv"
ts_out = "../data/processed/processed_timeseries_level.csv"

labeled_device_df.to_csv(device_out, index=False)
labeled_ts_df.to_csv(ts_out, index=False)

print(f"Saved: {device_out} | shape={labeled_device_df.shape}")
print(f"Saved: {ts_out} | shape={labeled_ts_df.shape}")

Saved: processed_device_level.csv | shape=(4149, 20)
Saved: processed_timeseries_level.csv | shape=(1514385, 10)
